# Latent gate: does the depth fix hold on held-out data and at 1B?

Runs `results/latent_gate/heldout_scale/PLAN.md`. Results so far: `docs/LATENT_GATE_STAGE1_RESULTS.md`.
- **Part A:** both GPT-2 models (SIM-CoT and CODI) on the held-out sets. Then `gate12_fixed`, the hard gate with its tie
  bug fixed, is trained and scored next to Stage 1's `ft6`.
- **Part B:** the LLaMA-3.2 1B models (CODI and SIM-CoT): discovery on validation, then the held-out sets.
- **Checks first, and the notebook stops if one fails.** The batched evaluator must match Stage 1's records (E1). The
  LLaMA port must pass its bias checks and reach within 3 points of the published GSM8K score (E2).

Every result is written to Drive as it is produced, so **rerunning after a disconnect resumes** where it stopped.

**Before running:**
- **Runtime:** Runtime → Change runtime type → **A100 GPU**, then Runtime → **Run all**.
- **Drive space:** about 1–2 GB, for two trained GPT-2 gates and the evaluations. Model downloads, including the 3.4 and
  6.4 GB 1B checkpoints, go to the Colab machine's own disk and are fetched again in each new session (a minute or two).
- **Time:** roughly 2–3 hours for Part A and 2–3 hours per 1B model. Running a few models per session is fine: trim
  `MODELS`, or let it stop and rerun later.
- **Keep the tab open** until the last cell downloads the results.

In [ ]:
# Settings. Rerunning with the same settings resumes; finished pieces are skipped.
MODELS = ["simcot-gpt2", "codi-gpt2", "codi-llama1b", "simcot-llama1b"]  # add "simcot-llama3b" only after both 1B models
TRAIN_GATE = True        # A2: train and score gate12_fixed on each GPT-2 model in MODELS
BATCH = {"gpt2": 64, "llama": 32}
REPO_REF = "main"        # the branch or commit holding scripts/latent_gate

In [ ]:
from pathlib import Path
import json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
OUT = DRIVE / "latent-gate" / "heldout-scale"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = OUT / "commit.txt"  # a started study keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert Path("scripts/latent_gate/heldout_eval.py").exists(), f"{ref} has no heldout_eval.py; push it first"
assert Path("results/latent_gate/heldout_scale/PLAN.sha256").exists(), "freeze the plan (PLAN.sha256) before running"
OUT.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
print("Code:", COMMIT, f"(from {ref})")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"  # local disk: the 1B checkpoints (3.4 and 6.4 GB) stay off Drive
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK = Path("/content/heldout-scale")  # fast local disk for training; finished models go to Drive
WORK.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=OUT / "logs")

run("scripts/setup_data.py", "--eval-only")  # includes the held-out sets
SIMCOT_GPT2 = DRIVE / "checkpoints/simcot-gpt2"
if "simcot-gpt2" in MODELS:
    run("scripts/latent_gate/fetch_simcot.py", "--output", SIMCOT_GPT2)

SIZES = {"gsm8k_valid": 500, "gsm8k_test": 1319, "gsm_hard": 1319, "multiarith": 180, "svamp": 1000}
HELDOUT = ["gsm8k_test", "gsm_hard", "multiarith", "svamp"]
HELDOUT_POLICIES = ["plain", "keep_first", "dedup_soft", "logn", "shift"]
DISCOVERY_POLICIES = ["plain", "keep_first", "window", "random", "logn", "dedup_soft", "shift", "copy_last", "copy_pair"]

def finished(folder, dataset):
    path = folder / f"{dataset}.jsonl"
    return path.exists() and len(path.read_text().splitlines()) == SIZES[dataset]

def evaluate(name, source, datasets, policies, *extra):
    """Score `source` (["--llama", x] or ["--checkpoint", dir] or []) into OUT/name, skipping finished datasets."""
    folder = OUT / name
    todo = [d for d in datasets if not finished(folder, d)]
    if not todo:
        print(f"{name}: {', '.join(datasets)} already on Drive")
        return
    family = "llama" if "--llama" in source else "gpt2"
    run("scripts/latent_gate/heldout_eval.py", *source, "--datasets", *todo, "--output-dir", folder,
        "--policies", *policies, "--batch", BATCH[family], "--device", "cuda", *extra)

def gpt2_source(model):
    return ["--checkpoint", SIMCOT_GPT2] if model == "simcot-gpt2" else []

## Part A: GPT-2

- **Discovery run.** Validation is scored with every policy and attention shares. Its overlap with Stage 1's batch-1
  records is check E1.
- **Held-out run.** The held-out sets are scored only after E1 passes.

In [ ]:
for model in [m for m in MODELS if m.endswith("gpt2")]:
    evaluate(model, gpt2_source(model), ["gsm8k_valid"], DISCOVERY_POLICIES, "--attention")
    stage1 = "simcot" if model == "simcot-gpt2" else "codi"
    run("scripts/latent_gate/check_batched.py", OUT / model / "gsm8k_valid.jsonl",
        f"results/latent_gate/stage1/{stage1}/evals/base/gsm8k_valid.jsonl")  # E1: raises if it fails
    evaluate(model, gpt2_source(model), HELDOUT, HELDOUT_POLICIES)

**A2.** The hard gate with its tie fix (`gate12_fixed`) gets Stage 1's data, order, seed and recipe. It is scored on
validation, then on the held-out sets beside Stage 1's `ft6` (`plain`), taken from that run's Drive backup.

In [ ]:
for model in [m for m in MODELS if m.endswith("gpt2")] if TRAIN_GATE else []:
    stage1 = "simcot" if model == "simcot-gpt2" else "codi"
    arm = OUT / "arms" / f"{model}-gate12_fixed"
    if not (arm / "model/gate.pt").exists():
        local = WORK / f"{model}-gate12_fixed"
        run("scripts/latent_gate/stage1_train.py", "--arm", "gate12_fixed", *(gpt2_source(model)), "--output", local,
            "--records", 32768, "--seed", 7, "--device", "cuda", "--bf16")
        shutil.copytree(local / "model", arm / "model", dirs_exist_ok=True)
        for name in ("args.json", "metrics.jsonl"):
            shutil.copy2(local / name, arm / name)
    evaluate(f"{model}-gate12_fixed", ["--checkpoint", arm / "model"], ["gsm8k_valid"], ["plain", "gate", "gate_soft"])
    evaluate(f"{model}-gate12_fixed", ["--checkpoint", arm / "model"], HELDOUT, ["gate"])
    ft6 = DRIVE / "latent-gate" / f"latent-gate-{stage1}-n32768-s7" / "arms" / "ft6" / "model"
    if ft6.exists():
        evaluate(f"{model}-ft6", ["--checkpoint", ft6], HELDOUT, ["plain"])
    else:
        print(f"{model}: Stage 1's ft6 is not on Drive at {ft6}; the A2 comparison with ft6 will be missing")

## Part B: LLaMA-3.2 1B (3B only if added to `MODELS`)

1. **Checks 1–2.** The bias checks on the real weights.
2. **Check 3.** The `plain` 6-step score on gsm8k_test, against the published one. Nothing else is scored on gsm8k_test
   before it passes.
3. **Discovery, then the held-out sets.**

In [ ]:
for model in [m for m in MODELS if "llama" in m]:
    done = OUT / model / "checks.txt"
    if not done.exists():
        run("scripts/latent_gate/check_llama.py", "--llama", model, "--device", "cuda")  # raises if it fails
        evaluate(f"{model}-published", ["--llama", model], ["gsm8k_test"], ["plain"], "--depths", 6)
        run("scripts/latent_gate/check_llama.py", "--llama", model, "--published", OUT / f"{model}-published" / "gsm8k_test.jsonl")
        done.parent.mkdir(parents=True, exist_ok=True)
        done.write_text("E2 checks 1-3 passed\n")
    evaluate(model, ["--llama", model], ["gsm8k_valid"], DISCOVERY_POLICIES, "--attention")
    evaluate(model, ["--llama", model], HELDOUT, HELDOUT_POLICIES)

In [ ]:
run("scripts/latent_gate/heldout_report.py", OUT)

In [ ]:
# The evaluations, gate training logs and logs, laid out for the repo's results/latent_gate/heldout_scale folder.
from google.colab import files

bundle = Path("/content/latent-gate-heldout-scale")
shutil.rmtree(bundle, ignore_errors=True)
for folder in OUT.iterdir():
    if folder.is_dir() and folder.name not in ("arms", "logs"):
        shutil.copytree(folder, bundle / folder.name)
for arm in (OUT / "arms").glob("*") if (OUT / "arms").exists() else []:
    for name in ("args.json", "metrics.jsonl"):
        if (arm / name).exists():
            (bundle / "training" / arm.name).mkdir(parents=True, exist_ok=True)
            shutil.copy2(arm / name, bundle / "training" / arm.name / name)
(bundle / "commit.txt").write_text(COMMIT)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, get it from the Files panel on the left.")

**Reading the result**
- **D2 is the main claim.** At 24 steps, soft dedup beats no eviction and stays within 2 points of the 6-step score, on
  the pooled held-out sets.
- **D1** asks whether the model loses accuracy past 6 steps at all. **D3** repeats D2 for keep-first-6. **D4** is the
  position-only control.
- **G** asks whether soft dedup at 12 or 24 steps beats 6 steps outright, which would be a gain.
- **The Part C trigger line** says whether training at 1B is worth running (plan, Part C).
- **A2** says whether the fixed gate holds accuracy from 12 to 24 steps and beats Stage 1's `ft6` at 24.

**Sharing:** the last cell downloads a zip of every evaluation. Point Claude at it.